In [ ]:
# Load R magic extension for Python Jupyter kernel (Kaggle / Colab support)
try:
    %load_ext rpy2.ipython
except Exception as e:
    print("Note on rpy2 initialization:", e)

# 4-Class Middle Tier & Extreme XGBoost Classifier (`models/xgboost_feng_esi234_extreme.ipynb`)

This notebook trains a **4-Class XGBoost Gradient Boosted Decision Tree Classifier** for **ESI 2 vs ESI 3 vs ESI 4 vs Extreme (ESI 1 & ESI 5)** using **13 Clinical Feature Engineered Predictors** with **Custom Per-Class Count & Ratio Control**:

### System Architecture & Workflow
1. **Full Patient Dataset (No Exclusion)**: Retains all ESI levels (ESI 1-5, $N \approx 560,000$ rows) and maps ESI 1 and ESI 5 into a single combined class `'extreme'`.
2. **4 Target Classes**: `"2"`, `"3"`, `"4"`, and `"extreme"`.
3. **Stratified Data Partitioning First**: Splits dataset into Train (70%), Validation (15%), and Test (15%) splits before scaling to prevent data leakage.
4. **Direct Custom Per-Class Sample Count & Ratio Control**: Independent knobs (`sample_ratio_esi2`, `sample_ratio_esi3`, `sample_ratio_esi4`, `sample_ratio_extreme`, or explicit `target_count_*`) on `train_df` to set sample counts/ratios for all 4 classes directly.
5. **Feature Set (13 Predictors)**: `age`, `gender`, `cc_breathingdifficulty`, and 10 vital sign anomaly flags (`is_dyspnea_total`, `is_dyspnea_moderate`, `is_bradypnea`, `is_tachypnea`, `is_hypotension`, `is_hypertension`, `is_bradycardia_total`, `is_bradycardia_moderate`, `is_tachycardia_total`, `is_tachycardia_moderate`).
6. **4-Class XGBoost Multi-Class Gradient Boosting**: Fits multi-class decision trees (`objective = "multi:softprob"`, `num_class = 4`, `eval_metric = "mlogloss"`) via `xgb.DMatrix` and `xgb.train()`.
7. **Comprehensive Benchmarking Across Splits**: Evaluates Train, Validation, and Test performance with 4x4 confusion matrices, target class count comparison tables across ESI 2, 3, 4, and Extreme, Accuracy, Macro Precision, Macro Recall, Macro F1, PR-AUC, and ROC-AUC.
8. **Reports & Artifacts**:
   - **Diagnostic Plots**: Metrics bar chart (`plots/xgboost_feng_esi234_metrics_barchart.png`).
   - **CSV Reports**: `reports/xgboost_feng_esi234_val_report.csv`, `reports/xgboost_feng_esi234_test_report.csv`.
   - **Model Export**: Saved to `deploy/xgboost_feng_esi234_extreme_model.rds`.

In [ ]:
%%R
# ---------------------------------------------------------
# Step 1: Load Required Libraries & Parse Configuration JSON
# ---------------------------------------------------------
library(jsonlite)
library(caret)
library(dplyr)
library(ggplot2)
library(tidyr)
library(pROC)
library(xgboost)
config_path <- "../config/triage_conf.json"
if (!file.exists(config_path)) {
  config_path <- "config/triage_conf.json"
}
config <- fromJSON(config_path)
cat("=== Configuration Loaded from config/triage_conf.json ===\n")
cat("Data Source Path:", config$path$data_source, "\n")
cat("Target Column:   ", config$classes$target_col, "\n")
cat("Test Size:       ", config$training$test_size, "\n")
cat("Val Size:        ", config$training$val_size, "\n")
cat("Random State:    ", config$training$random_state, "\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 2: Load Data, Retain Full Dataset (ESI 1-5), Map ESI 1 & 5 to 'extreme'
# ---------------------------------------------------------
set.seed(config$training$random_state)
data_file <- config$path$data_source
if (!file.exists(data_file) && file.exists(paste0("../", data_file))) {
  data_file <- paste0("../", data_file)
}
cat("Loading dataset from:", data_file, "...\n")
data_env <- new.env()
load(data_file, envir = data_env)
df_names <- ls(data_env)[sapply(ls(data_env), function(x) is.data.frame(get(x, envir = data_env)))]
df_sizes <- sapply(df_names, function(x) nrow(get(x, envir = data_env)))
data_obj_name <- df_names[which.max(df_sizes)]
cat(sprintf("Selected main dataset object: '%s' (%d rows)\n", data_obj_name, max(df_sizes)))
raw_df <- get(data_obj_name, envir = data_env)
target_col <- config$classes$target_col
raw_esi <- as.character(raw_df[[target_col]])
cat(sprintf("Retaining Full Dataset: %d total rows (ESI 1 and ESI 5 grouped into 'extreme')\n", nrow(raw_df)))
gender_vec <- if ("gender" %in% names(raw_df)) ifelse(as.character(raw_df$gender) == "Male", 1, 0) else 0
cc_bd_vec  <- if ("cc_breathingdifficulty" %in% names(raw_df)) ifelse(!is.na(raw_df$cc_breathingdifficulty), raw_df$cc_breathingdifficulty, 0) else 0
# Construct 13 Clinical Feature Engineering flags
df_feng <- data.frame(
  age                     = raw_df$age,
  gender                  = gender_vec,
  cc_breathingdifficulty  = cc_bd_vec,
  is_dyspnea_total        = ifelse(!is.na(raw_df$triage_vital_o2) & raw_df$triage_vital_o2 < 90, 1, 0),
  is_dyspnea_moderate     = ifelse(!is.na(raw_df$triage_vital_o2) & raw_df$triage_vital_o2 >= 90 & raw_df$triage_vital_o2 < 94, 1, 0),
  is_bradypnea            = ifelse(!is.na(raw_df$triage_vital_rr) & raw_df$triage_vital_rr < 10, 1, 0),
  is_tachypnea            = ifelse(!is.na(raw_df$triage_vital_rr) & raw_df$triage_vital_rr > 30, 1, 0),
  is_hypotension          = ifelse(!is.na(raw_df$triage_vital_sbp) & raw_df$triage_vital_sbp <= 90, 1, 0),
  is_hypertension         = ifelse(!is.na(raw_df$triage_vital_sbp) & raw_df$triage_vital_sbp > 220, 1, 0),
  is_bradycardia_total    = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr < 40, 1, 0),
  is_bradycardia_moderate = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr >= 40 & raw_df$triage_vital_hr < 60, 1, 0),
  is_tachycardia_total    = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr > 150, 1, 0),
  is_tachycardia_moderate = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr > 100 & raw_df$triage_vital_hr <= 150, 1, 0)
)
# MAP ESI 1 AND 5 TO 'extreme', KEEP 2, 3, 4 AS IS
target_vec <- ifelse(raw_esi %in% c("1", "5"), "extreme", raw_esi)
df_feng$target_4class <- factor(target_vec, levels = c("2", "3", "4", "extreme"))
initial_rows <- nrow(df_feng)
df_feng <- na.omit(df_feng)
cat(sprintf("Complete Case Filtering: Removed %d rows with NULL/NA features (Remaining complete rows: %d)\n",
            initial_rows - nrow(df_feng), nrow(df_feng)))
cat(sprintf("Full 4-Class Dataset Ready (Pre-Partitioning): %d total rows x %d cols\n", nrow(df_feng), ncol(df_feng)))
cat("Natural 4-Class Target Distribution ('2', '3', '4', 'extreme'):\n")
print(table(df_feng$target_4class))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 3: Stratified Data Partitioning FIRST & Custom Per-Class Sample Control for 4 Classes
# ---------------------------------------------------------
set.seed(config$training$random_state)
# DIRECT CUSTOM TRAINING SAMPLE COUNT / RATIO PER CLASS FOR 4 CLASSES
# Option A: Set explicit target row counts per class (or NULL to use ratio-based scaling)
target_count_esi2    <- NULL  # e.g., 50000, 80000, or NULL
target_count_esi3    <- NULL  # e.g., 50000, 80000, or NULL
target_count_esi4    <- NULL  # e.g., 50000, 80000, or NULL
target_count_extreme <- NULL  # e.g., 50000, 80000, or NULL
# Option B: Set custom ratios relative to ESI 4 (used when target_count_* is NULL)
# Set 1.0 : 1.0 : 1.0 : 1.0 for 1:1:1:1 balanced ratio across 4 classes, or adjust independently
sample_ratio_esi2    <- 1.0
sample_ratio_esi3    <- 1.0
sample_ratio_esi4    <- 1.0
sample_ratio_extreme <- 1.0
test_size <- config$training$test_size
val_size  <- config$training$val_size
# Stratified Test split (15%)
in_train_val <- createDataPartition(df_feng$target_4class, p = 1 - test_size, list = FALSE)
train_val_df <- df_feng[in_train_val, ]
test_df      <- df_feng[-in_train_val, ]
# Stratified Validation split (15%)
rel_val_size <- val_size / (1 - test_size)
in_train    <- createDataPartition(train_val_df$target_4class, p = 1 - rel_val_size, list = FALSE)
train_df    <- train_val_df[in_train, ]
val_df      <- train_val_df[-in_train, ]
# Standardize continuous feature (age)
cont_cols <- "age"
preproc <- preProcess(train_df[, cont_cols, drop = FALSE], method = c("center", "scale"))
train_df <- predict(preproc, train_df)
val_df   <- predict(preproc, val_df)
test_df  <- predict(preproc, test_df)
cat(sprintf("Pre-sampling Partition sizes:\n  Train: %d rows\n  Val:   %d rows\n  Test:  %d rows\n\n",
            nrow(train_df), nrow(val_df), nrow(test_df)))
# ---------------------------------------------------------
# APPLY DIRECT CUSTOM PER-CLASS COUNT & RATIO CONTROL TO TRAINING SET ONLY
# ---------------------------------------------------------
idx_2_tr       <- which(train_df$target_4class == "2")
idx_3_tr       <- which(train_df$target_4class == "3")
idx_4_tr       <- which(train_df$target_4class == "4")
idx_extreme_tr <- which(train_df$target_4class == "extreme")
base_n4 <- length(idx_4_tr)
n2_keep       <- if (!is.null(target_count_esi2)) min(target_count_esi2, length(idx_2_tr)) else min(as.integer(base_n4 * sample_ratio_esi2), length(idx_2_tr))
n3_keep       <- if (!is.null(target_count_esi3)) min(target_count_esi3, length(idx_3_tr)) else min(as.integer(base_n4 * sample_ratio_esi3), length(idx_3_tr))
n4_keep       <- if (!is.null(target_count_esi4)) min(target_count_esi4, length(idx_4_tr)) else min(as.integer(base_n4 * sample_ratio_esi4), length(idx_4_tr))
nextreme_keep <- if (!is.null(target_count_extreme)) min(target_count_extreme, length(idx_extreme_tr)) else min(as.integer(base_n4 * sample_ratio_extreme), length(idx_extreme_tr))
kept_2       <- sample(idx_2_tr, size = n2_keep)
kept_3       <- sample(idx_3_tr, size = n3_keep)
kept_4       <- sample(idx_4_tr, size = n4_keep)
kept_extreme <- sample(idx_extreme_tr, size = nextreme_keep)
train_df <- train_df[sample(c(kept_2, kept_3, kept_4, kept_extreme)), ]
cat("=== Custom Per-Class Sample Control Applied to Training Set ===\n")
cat(sprintf("  - ESI 2 Count:   %d rows (Ratio: %.2f)\n", length(kept_2), sample_ratio_esi2))
cat(sprintf("  - ESI 3 Count:   %d rows (Ratio: %.2f)\n", length(kept_3), sample_ratio_esi3))
cat(sprintf("  - ESI 4 Count:   %d rows (Ratio: %.2f)\n", length(kept_4), sample_ratio_esi4))
cat(sprintf("  - Extreme Count: %d rows (Ratio: %.2f)\n", length(kept_extreme), sample_ratio_extreme))
cat(sprintf("  - Total Training Size: %d rows\n\n", nrow(train_df)))
cat("Custom Target Distribution in Training Set ('2', '3', '4', 'extreme'):\n")
print(table(train_df$target_4class))
cat("\nNatural Validation Set Target Distribution:\n")
print(table(val_df$target_4class))
cat("\nNatural Test Set Target Distribution:\n")
print(table(test_df$target_4class))
feat_names <- setdiff(names(train_df), "target_4class")
X_train <- as.matrix(train_df[, feat_names])
y_train_num <- as.numeric(train_df$target_4class) - 1  # 0, 1, 2, 3 for classes '2', '3', '4', 'extreme'
X_val   <- as.matrix(val_df[, feat_names])
y_val_num   <- as.numeric(val_df$target_4class) - 1
X_test  <- as.matrix(test_df[, feat_names])
y_test_num  <- as.numeric(test_df$target_4class) - 1
dtrain_xgb <- xgb.DMatrix(data = X_train, label = y_train_num)
dval_xgb   <- xgb.DMatrix(data = X_val,   label = y_val_num)
dtest_xgb  <- xgb.DMatrix(data = X_test,  label = y_test_num)

In [ ]:
%%R
# ---------------------------------------------------------
# Step 4: Train 4-Class XGBoost Model on Sample-Regulated Data
# ---------------------------------------------------------
set.seed(config$training$random_state)
cat("Training 4-Class XGBoost Model ('2' vs '3' vs '4' vs 'extreme')...\n")
xgb_params <- list(
  objective        = "multi:softprob",
  num_class        = 4,
  eval_metric      = "mlogloss",
  eta              = 0.05,
  max_depth        = 6,
  subsample        = 0.8,
  colsample_bytree = 0.8
)
xgb_model <- xgb.train(
  params    = xgb_params,
  data      = dtrain_xgb,
  nrounds   = 150,
  evals     = list(train = dtrain_xgb, val = dval_xgb),
  early_stopping_rounds = 20,
  verbose   = 0
)
cat("XGBoost 4-Class Training Complete!\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 5: Comprehensive Benchmark Across Splits & Export CSV Reports
# ---------------------------------------------------------
calc_pr_auc <- function(actual_binary, prob_positive) {
  tryCatch({
    ord <- order(prob_positive, decreasing = TRUE)
    act_sorted <- (actual_binary[ord] == 1)
    tp <- cumsum(act_sorted)
    fp <- cumsum(!act_sorted)
    n_pos <- sum(act_sorted)
    if (n_pos == 0) return(NA)
    rec <- c(0, tp / n_pos)
    prec <- c(tp[1] / max(1, tp[1] + fp[1]), tp / (tp + fp))
    dx <- diff(rec)
    my <- (prec[-1] + prec[-length(prec)]) / 2
    return(as.numeric(sum(dx * my)))
  }, error = function(e) NA)
}
evaluate_xgboost_4class <- function(model, dmatrix, actual_factor, set_name) {
  target_classes <- c("2", "3", "4", "extreme")
  
  prob_raw <- predict(model, newdata = dmatrix)
  prob_matrix <- matrix(prob_raw, ncol = 4, byrow = TRUE)
  colnames(prob_matrix) <- target_classes
  
  max_idx <- max.col(prob_matrix, ties.method = "first")
  pred_fac <- factor(target_classes[max_idx], levels = target_classes)
  act_fac  <- factor(actual_factor, levels = target_classes)
  
  cm  <- confusionMatrix(pred_fac, act_fac)
  acc <- as.numeric(cm$overall["Accuracy"])
  prec_by_class <- cm$byClass[, "Pos Pred Value"]
  rec_by_class  <- cm$byClass[, "Sensitivity"]
  macro_prec    <- mean(prec_by_class, na.rm = TRUE)
  macro_rec     <- mean(rec_by_class,  na.rm = TRUE)
  macro_f1      <- 2 * (macro_prec * macro_rec) / (macro_prec + macro_rec + 1e-15)
  
  pr_auc_by_class <- numeric(4)
  names(pr_auc_by_class) <- target_classes
  for (cls in target_classes) {
    act_bin <- ifelse(act_fac == cls, 1, 0)
    pr_auc_by_class[cls] <- calc_pr_auc(act_bin, prob_matrix[, cls])
  }
  macro_pr_auc <- mean(pr_auc_by_class, na.rm = TRUE)
  
  roc_obj <- tryCatch(pROC::multiclass.roc(act_fac, prob_matrix), error = function(e) NULL)
  macro_roc_auc <- if (!is.null(roc_obj)) as.numeric(roc_obj$auc) else NA
  
  actual_table <- table(act_fac)
  pred_table   <- table(pred_fac)
  diff_vec     <- as.numeric(pred_table) - as.numeric(actual_table)
  diff_str     <- ifelse(diff_vec >= 0, paste0("+", diff_vec), as.character(diff_vec))
  
  report_df <- data.frame(
    Class        = target_classes,
    Actual_Count = as.numeric(actual_table),
    Pred_Count   = as.numeric(pred_table),
    Diff         = diff_str,
    Precision    = round(prec_by_class, 4),
    Recall       = round(rec_by_class, 4),
    PR_AUC       = round(pr_auc_by_class, 4)
  )
  
  cat(sprintf("============================================================\n"))
  cat(sprintf("   4-CLASS XGBOOST (2, 3, 4, EXTREME) - %s SET BENCHMARK\n", toupper(set_name)))
  cat(sprintf("============================================================\n"))
  cat(sprintf("  Overall Accuracy     : %.4f (%.2f%%)\n", acc, acc * 100))
  cat(sprintf("  Macro Precision      : %.4f (%.2f%%)\n", macro_prec, macro_prec * 100))
  cat(sprintf("  Macro Recall (Sens)  : %.4f (%.2f%%)\n", macro_rec, macro_rec * 100))
  cat(sprintf("  Macro F1 Score       : %.4f\n", macro_f1))
  cat(sprintf("  Macro PR-AUC         : %.4f\n", macro_pr_auc))
  cat(sprintf("  Multi-Class ROC-AUC  : %.4f\n", macro_roc_auc))
  cat(sprintf("============================================================\n\n"))
  
  cat("Class Count Comparison & Performance Summary:\n")
  print(report_df)
  cat("\n4x4 Confusion Matrix (Rows: Predicted, Columns: Actual):\n")
  print(cm$table)
  cat(sprintf("============================================================\n\n"))
  
  return(list(acc = acc, macro_prec = macro_prec, macro_rec = macro_rec, macro_f1 = macro_f1, macro_pr_auc = macro_pr_auc, macro_roc_auc = macro_roc_auc, report_df = report_df))
}
res_train <- evaluate_xgboost_4class(xgb_model, dtrain_xgb, train_df$target_4class, "Train")
res_val   <- evaluate_xgboost_4class(xgb_model, dval_xgb,   val_df$target_4class,   "Validation")
res_test  <- evaluate_xgboost_4class(xgb_model, dtest_xgb,  test_df$target_4class,  "Test")
# Write CSV Reports
reports_dir <- "../reports"
if (!dir.exists(reports_dir)) reports_dir <- "reports"
if (!dir.exists(reports_dir)) dir.create(reports_dir, recursive = TRUE)
write.csv(res_val$report_df,  file = file.path(reports_dir, "xgboost_feng_esi234_val_report.csv"),  row.names = FALSE)
write.csv(res_test$report_df, file = file.path(reports_dir, "xgboost_feng_esi234_test_report.csv"), row.names = FALSE)
cat("Validation CSV Report written to: reports/xgboost_feng_esi234_val_report.csv\n")
cat("Test CSV Report written to:       reports/xgboost_feng_esi234_test_report.csv\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 6: Diagnostic Plots (Metrics Bar Chart)
# ---------------------------------------------------------
plots_dir <- "../plots"
if (!dir.exists(plots_dir)) plots_dir <- "plots"
if (!dir.exists(plots_dir)) dir.create(plots_dir, recursive = TRUE)
metrics_summary <- data.frame(
  Split     = factor(c("Train", "Validation", "Test"), levels = c("Train", "Validation", "Test")),
  Accuracy  = c(res_train$acc,          res_val$acc,          res_test$acc),
  Precision = c(res_train$macro_prec,   res_val$macro_prec,   res_test$macro_prec),
  Recall    = c(res_train$macro_rec,    res_val$macro_rec,    res_test$macro_rec),
  F1_Score  = c(res_train$macro_f1,     res_val$macro_f1,     res_test$macro_f1),
  PR_AUC    = c(res_train$macro_pr_auc, res_val$macro_pr_auc, res_test$macro_pr_auc)
)
metrics_long <- metrics_summary %>%
  pivot_longer(cols = c("Accuracy", "Precision", "Recall", "F1_Score", "PR_AUC"), names_to = "Metric", values_to = "Score")
p_bar <- ggplot(metrics_long, aes(x = Metric, y = Score, fill = Split)) +
  geom_bar(stat = "identity", position = position_dodge(width = 0.7), width = 0.6) +
  geom_text(aes(label = sprintf("%.3f", Score)), position = position_dodge(width = 0.7), vjust = -0.3, size = 3) +
  theme_minimal() +
  scale_fill_manual(values = c("Train" = "#2b5c8f", "Validation" = "#e07a5f", "Test" = "#81b29a")) +
  labs(title = "Train vs. Validation vs. Test Metrics Comparison (4-Class ESI 2, 3, 4, Extreme XGBoost)",
       subtitle = "Comparing Accuracy, Precision, Recall, F1 Score, and PR-AUC across splits",
       y = "Metric Value Score", x = "") +
  theme(plot.title = element_text(face = "bold", size = 13), legend.position = "top")
ggsave(file.path(plots_dir, "xgboost_feng_esi234_metrics_barchart.png"), plot = p_bar, width = 9.5, height = 5, dpi = 300)
cat("Metrics Comparison Bar Chart saved to: plots/xgboost_feng_esi234_metrics_barchart.png\n")
p_bar

In [ ]:
%%R
# ---------------------------------------------------------
# Step 7: Save XGBoost 4-Class Model Artifact
# ---------------------------------------------------------
deploy_dir <- "../deploy"
if (!dir.exists(deploy_dir)) deploy_dir <- "deploy"
if (!dir.exists(deploy_dir)) dir.create(deploy_dir, recursive = TRUE)
model_path <- file.path(deploy_dir, "xgboost_feng_esi234_extreme_model.rds")
saveRDS(list(model = xgb_model, preproc = preproc), file = model_path)
cat("4-Class ESI 2, 3, 4, Extreme XGBoost model saved to:", model_path, "\n")